# ForgeLM — Document Ingestion Playground

Upload your own PDF / DOCX / EPUB / TXT / Markdown files, run
`forgelm ingest` to chunk + (optionally) mask PII / secrets, audit
the result with `forgelm audit`, and download the curated JSONL + audit
report as a single ZIP.

**Why use this notebook:**
- Quick sanity check on a corpus before you commit to a full training run.
- See exactly what chunks the model will train on (heading breadcrumbs, redactions, fence handling).
- Pull out an EU AI Act Article 10 governance artefact (`data_audit_report.json`) without writing any YAML.

**Requirements:** any runtime — no GPU needed. Both `ingest` and `audit` are CPU-bound and stream their input.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/ingestion_playground.ipynb)


In [ ]:
# Step 1: Install ForgeLM with the ingestion extra (PDF/DOCX/EPUB parsers + langdetect)
# Pinned to v0.5.7; bump on each release.
!pip install -q --no-cache-dir 'forgelm[ingestion]==0.5.7'
!forgelm --version


## 1. Upload your documents

Supported extensions: `.pdf`, `.docx`, `.epub`, `.txt`, `.md`.
Anything else is silently skipped by the ingester.

The cell below auto-detects Colab vs local Jupyter:

- **Colab** — opens the native file picker (multi-select supported).
- **Local** — drop files into the `./raw_docs/` folder beside this notebook and just run the cell to inventory them.


In [ ]:
import shutil
from pathlib import Path

raw_dir = Path("./raw_docs")
raw_dir.mkdir(exist_ok=True)

SUPPORTED = {".pdf", ".docx", ".epub", ".txt", ".md"}

# Detect Colab; fall back to local-dir mode otherwise.
try:
    from google.colab import files as colab_files  # type: ignore
    in_colab = True
except ImportError:
    colab_files = None
    in_colab = False

if in_colab:
    print("Colab detected. Pick one or more documents — pdf / docx / epub / txt / md.")
    uploaded = colab_files.upload()  # opens the file picker; user selects file(s)
    for name, data in uploaded.items():
        dest = raw_dir / name
        dest.write_bytes(data)
    print(f"\nMoved {len(uploaded)} file(s) into {raw_dir.resolve()}/")
else:
    print(f"Local Jupyter detected. Drop your documents into {raw_dir.resolve()}/ then re-run this cell.")

# Inventory whatever ended up in raw_docs/
found = sorted(p for p in raw_dir.rglob("*") if p.is_file())
if not found:
    print("\nNo files in raw_docs/ yet. Upload (Colab) or copy in (local) and re-run.")
else:
    print(f"\nFiles ready for ingestion ({len(found)}):")
    for f in found:
        marker = "  " if f.suffix.lower() in SUPPORTED else "  ! "
        suffix_note = "" if f.suffix.lower() in SUPPORTED else f"  (unsupported — will be skipped)"
        print(f"{marker}{f.relative_to(raw_dir)}  {f.stat().st_size:>8} bytes{suffix_note}")


## 2. Configure ingestion

Edit the variables below to taste, then run the cell.

| Variable | What it does |
|---|---|
| `STRATEGY` | `"paragraph"` (default) / `"sliding"` / `"markdown"`. Markdown is heading-aware and best for `.md` corpora. Paragraph is the safe default for mixed corpora. |
| `CHUNK_SIZE` | Soft per-chunk character cap (default 2048). 600–1024 is good for tight LMs (≤4k context); 4096 for bigger LMs. |
| `MASK_PII` | Redact names / emails / phone numbers / IDs before writing. |
| `MASK_SECRETS` | Redact API keys / tokens / SSH keys / connection strings. |
| `RECURSIVE` | Descend into subdirectories of `./raw_docs/`. |

Mask-on-by-default keeps your training data PR-able — turn them off only if you've already cleaned upstream.


In [ ]:
# Tweak these and re-run the cell. The next cell calls `forgelm ingest` with these values.
STRATEGY = "paragraph"     # "paragraph" | "sliding" | "markdown"
CHUNK_SIZE = 1024          # soft character cap per chunk
MASK_PII = True            # redact PII (names, emails, phones, IDs)
MASK_SECRETS = True        # redact API keys, tokens, SSH/PGP keys, JWTs
RECURSIVE = True           # descend into subdirectories
OUTPUT_PATH = "./curated/dataset.jsonl"  # where the curated JSONL lands

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

# Build the CLI invocation
flags = [f"--strategy {STRATEGY}", f"--chunk-size {CHUNK_SIZE}", f"--output {OUTPUT_PATH}", "--output-format json"]
if RECURSIVE:
    flags.append("--recursive")
if MASK_PII:
    flags.append("--pii-mask")
if MASK_SECRETS:
    flags.append("--secrets-mask")
cmd = "forgelm ingest ./raw_docs/ " + " ".join(flags)
print("Running:\n  " + cmd + "\n")


In [ ]:
# Step 3: Run the ingestion.  Reads ./raw_docs/, writes the curated JSONL.
# (The variables from the previous cell are interpolated by the IPython shell magic.)
!forgelm ingest ./raw_docs/ \
    {'--recursive' if RECURSIVE else ''} \
    --strategy {STRATEGY} \
    --chunk-size {CHUNK_SIZE} \
    {'--pii-mask' if MASK_PII else ''} \
    {'--secrets-mask' if MASK_SECRETS else ''} \
    --output {OUTPUT_PATH} \
    --output-format json


In [ ]:
# Step 4: Preview the chunks ForgeLM produced.
import json
from pathlib import Path

out = Path(OUTPUT_PATH)
if not out.is_file():
    print(f"No output at {out}. Did `forgelm ingest` finish without errors? Check the previous cell.")
else:
    rows = out.read_text(encoding="utf-8").splitlines()
    print(f"Total chunks: {len(rows)}")
    print(f"File size:    {out.stat().st_size:,} bytes\n")
    PREVIEW_CHUNKS = 5  # change to inspect more / fewer
    for i, line in enumerate(rows[:PREVIEW_CHUNKS]):
        chunk = json.loads(line)["text"]
        print(f"=== Chunk {i} ({len(chunk)} chars) ===")
        print(chunk[:600] + ("…" if len(chunk) > 600 else ""))
        print()
    if len(rows) > PREVIEW_CHUNKS:
        print(f"... {len(rows) - PREVIEW_CHUNKS} more chunk(s) in the JSONL.")


## 3. Audit the curated corpus (optional but recommended)

`forgelm audit` walks the JSONL and produces a `data_audit_report.json`
covering:

- per-split sample count + length distribution
- top-3 detected languages (via `langdetect`)
- near-duplicate scan (simhash by default, LSH-banded)
- PII severity tiers (`critical` / `high` / `medium` / `low`)
- always-on secrets pass — flags credentials that survived ingestion

If you ran ingestion with `--secrets-mask`, the secrets count here
should be **zero** — that's the audit's job, to confirm the masker did
what it said it did.


In [ ]:
# Step 5: Audit the curated JSONL.
AUDIT_DIR = "./audit"
!forgelm audit {OUTPUT_PATH} --output {AUDIT_DIR} --output-format json


In [ ]:
# Step 6: Pretty-print the audit highlights (the on-disk JSON carries the full schema).
import json
from pathlib import Path

report_path = Path(AUDIT_DIR) / "data_audit_report.json"
if not report_path.is_file():
    print(f"No audit report at {report_path}. Did `forgelm audit` finish without errors?")
else:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("AUDIT SUMMARY")
    print("=" * 50)
    print(f"Total samples:    {report.get('total_samples')}")
    print(f"Splits:           {list(report.get('splits', {}).keys())}")

    near = report.get("near_duplicate_summary", {})
    if near:
        print(f"\nNear-duplicate method: {near.get('method')}")
        print(f"  Pairs per split:     {near.get('pairs_per_split')}")

    pii = report.get("pii_summary", {})
    sev = report.get("pii_severity", {})
    if pii:
        print(f"\nPII spans:        {pii}")
        if sev:
            print(f"  Worst tier:     {sev.get('worst_tier')}")

    secrets = report.get("secrets_summary", {})
    if secrets:
        print(f"\nSecrets summary (POST-MASK — should be empty if --secrets-mask was on):")
        print(f"  {secrets}")
    else:
        print("\nSecrets:          clean (no credentials reached the JSONL).")

    lang = report.get("language_summary", {})
    if lang:
        print(f"\nLanguage top-3:   {lang}")


## 4. Download everything

Bundles `curated/dataset.jsonl` + `audit/data_audit_report.json` into
a single ZIP. The download cell auto-detects Colab and triggers the
browser download; on local Jupyter, the ZIP just lands beside this
notebook and you grab it from your filesystem.


In [ ]:
import shutil
from pathlib import Path

ZIP_BASE = "forgelm_ingestion_output"
ZIP_PATH = Path(f"{ZIP_BASE}.zip")

# Build a clean staging dir so the ZIP has a tidy top-level layout.
stage = Path("./_export_staging")
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()

if Path(OUTPUT_PATH).is_file():
    shutil.copy(OUTPUT_PATH, stage / Path(OUTPUT_PATH).name)

audit_root = Path(AUDIT_DIR)
if audit_root.is_dir():
    shutil.copytree(audit_root, stage / audit_root.name)

# shutil.make_archive returns the absolute path of the produced .zip.
archive_path = shutil.make_archive(ZIP_BASE, "zip", root_dir=stage)
shutil.rmtree(stage)
print(f"Created: {archive_path}  ({Path(archive_path).stat().st_size:,} bytes)")

# Try Colab download; fall back to a filesystem path hint.
try:
    from google.colab import files as colab_files  # type: ignore
    colab_files.download(archive_path)
except ImportError:
    print(f"Local Jupyter: download the ZIP from {Path(archive_path).resolve()}")


## What's next

- **Train on this curated JSONL.** Point any training notebook's
  `data.dataset_name_or_path` at `./curated/dataset.jsonl` — see
  [`quickstart_sft.ipynb`](./quickstart_sft.ipynb) for the smallest
  SFT loop you can run on Colab.
- **Need bigger corpora?** Above ~50K rows, switch to MinHash LSH
  dedup: re-run `forgelm audit` with
  `--dedup-method minhash --jaccard-threshold 0.85` (the
  `forgelm[ingestion-scale]` extra adds `datasketch`).
- **Token-aware chunking.** If your model context is the real cap
  (not characters), swap `--chunk-size N` for `--chunk-tokens N
  --tokenizer HF_ID` in Step 3.
- **Long-form guides:** [`docs/guides/ingestion.md`](https://github.com/cemililik/ForgeLM/blob/main/docs/guides/ingestion.md)
  and [`docs/guides/data_audit.md`](https://github.com/cemililik/ForgeLM/blob/main/docs/guides/data_audit.md)
  cover every flag, every chunking strategy, and the full audit
  JSON schema.
